# Data Behavior Refactor Audit

This notebook compares datamodule/sample behavior between two separate LFM checkouts, for example:

- base repo: `ibm_model`
- refactor repo: `ibm_oop_refactor`

It intentionally runs each repo audit in a separate Python subprocess so imports from one checkout cannot pollute the other checkout through `sys.modules`.

The audit does not train. It instantiates configured datamodules, samples a few items from train/val/test, summarizes tensors/paths/targets, and compares JSON summaries.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import tempfile
from pprint import pprint

# Edit these paths before running.
BASE_REPO_ROOT = Path(r"/explore/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm_old/lfm")
OOP_REPO_ROOT = Path(r"/explore/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm")
DATA_ROOT = Path(r"/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")

# Usually both repos should read the same data root. Override these only if the old branch
# needs a different prepared data layout.
BASE_DATA_ROOT = DATA_ROOT
OOP_DATA_ROOT = DATA_ROOT

# Use the same Python executable for both repos unless you need different envs.
BASE_PYTHON = sys.executable
OOP_PYTHON = sys.executable

AUDIT_OUTPUT_ROOT = Path("/explore/nobackup/people/ajkerr1/Lunar_FM") / "lfm_data_behavior_audit_outputs"
AUDIT_RUN_NAME = datetime.now().strftime("date_%Y_%m_%d-time_%H_%M_%S")
OUTPUT_DIR = AUDIT_OUTPUT_ROOT / AUDIT_RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this small. This is intended to catch preprocessing/target contract drift, not benchmark training.
N_SAMPLES_PER_SPLIT = 3
AUDIT_CASE_TIMEOUT_SECONDS = 300
TARGET_SIZE = 256
IMAGE_GLOB = "*.tif"
LABEL_GLOB = "*_label.npz"
IMAGE_SUFFIX = "_input_wac_chip"
LABEL_SUFFIX = "_label"

for helper_dir in (Path.cwd(), Path.cwd() / "notebooks" / "full_model"):
    if (helper_dir / "data_behavior_refactor_audit_utils.py").exists():
        sys.path.insert(0, str(helper_dir))
        break
else:
    raise FileNotFoundError("Could not find data_behavior_refactor_audit_utils.py")

from data_behavior_refactor_audit_utils import (
    build_audit_cases,
    compare_audit_results,
    enabled_audit_cases,
    run_audit_cases,
    write_runner,
)

print("base repo:", BASE_REPO_ROOT)
print("oop repo:", OOP_REPO_ROOT)
print("base data root:", BASE_DATA_ROOT)
print("oop data root:", OOP_DATA_ROOT)
print("audit output root:", AUDIT_OUTPUT_ROOT)
print("audit run name:", AUDIT_RUN_NAME)
print("output dir:", OUTPUT_DIR)

## Audit Cases

Each case names a datamodule class and constructor kwargs. If the base branch has older module paths, edit the `base_module` / `base_class` fields for that case without changing the refactor side.

If one side needs different constructor kwargs, add `base_kwargs` or `oop_kwargs` to that case. Those override `kwargs` for only that side.

Disable cases that do not apply to your dataset by setting `enabled: False`.

In [ ]:
AUDIT_CASES = build_audit_cases(
    TARGET_SIZE,
    image_glob=IMAGE_GLOB,
    label_glob=LABEL_GLOB,
    image_suffix=IMAGE_SUFFIX,
    label_suffix=LABEL_SUFFIX,
)
enabled_cases = enabled_audit_cases(AUDIT_CASES)
print("enabled cases:", [case["name"] for case in enabled_cases])

In [ ]:
RUNNER_PATH = write_runner(OUTPUT_DIR)
print("runner:", RUNNER_PATH)

In [ ]:
audit_results = run_audit_cases(
    audit_cases=enabled_cases,
    base_repo_root=BASE_REPO_ROOT,
    oop_repo_root=OOP_REPO_ROOT,
    base_data_root=BASE_DATA_ROOT,
    oop_data_root=OOP_DATA_ROOT,
    base_python=BASE_PYTHON,
    oop_python=OOP_PYTHON,
    runner_path=RUNNER_PATH,
    n_samples=N_SAMPLES_PER_SPLIT,
    timeout_seconds=AUDIT_CASE_TIMEOUT_SECONDS,
)

raw_path = OUTPUT_DIR / "audit_raw_results.json"
raw_path.write_text(json.dumps(audit_results, indent=2, sort_keys=True), encoding="utf-8")
print("wrote", raw_path)

In [ ]:
comparison_rows = compare_audit_results(audit_results, enabled_cases)

comparison_path = OUTPUT_DIR / "audit_differences.json"
comparison_path.write_text(json.dumps(comparison_rows, indent=2, sort_keys=True), encoding="utf-8")
print(f"differences: {len(comparison_rows)}")
print("wrote", comparison_path)
pprint(comparison_rows[:50])

## How To Read Results

- `audit_raw_results.json` contains the full per-repo summaries.
- `audit_differences.json` contains only mismatched flattened fields.
- Differences in `module`, `class`, `datamodule_type`, and split `dataset_type` are ignored by default because those are expected to change during refactors.
- Path strings may differ if the two checkouts are in different folders; filename fields are usually more useful than full path fields.
- For Phase 2/3 behavior preservation, prioritize differences in shapes, channel counts, image min/max/mean/std, mask unique values, object counts, box shapes, and batch keys.